In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print(os.listdir("../input"))

# Any results you write to the current directory are saved as output.

# Titanic with decision trees

Using sklearn's DecisionTreeClassifier to create a tree image for surviving the titanic

In [2]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.model_selection import train_test_split

import graphviz
from subprocess import check_call

from IPython.display import Image as PImage
from PIL import Image, ImageDraw, ImageFont

In [12]:
df = pd.read_csv('../input/train.csv', error_bad_lines=False)

In [13]:
df.head(3)

Before we go on, let's do a bit of data prep.

Let's just drop any NaNs - there are other options of course for NaNs, but we're interested in creating a decision tree not creating the most accurate classifier ever

In [15]:
df = df.dropna(axis=1)
df = df.drop(['Name', 'Ticket'], axis=1)

Now the decision tree we're going to use doesn't work well with categorical data, so lets use `pandas`' `get_dummies` function to create a one hot encoding of the troublesome columns

We will also split the data into training and test sets

In [16]:
X = pd.get_dummies(df.drop('Survived', axis=1))
Y = pd.get_dummies(df['Survived'])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=42)

Just out of interest, this is how `get_dummies` changes the data

In [17]:
X_train.head(3)

So let's train a classifier!

As you can see below we have set the `max_depth=3` and `min_samples_split=20` for this classifier. That's to prevent overfitting

In [18]:
clf = DecisionTreeClassifier(max_depth=3, min_samples_split=20)
clf = clf.fit(X_train, y_train)

In [19]:
clf.score(X_test, y_test)

Great stuff! let's vizualise this

In [20]:
with open("tree1.dot", 'w') as f:
     f = export_graphviz(clf,
                              out_file=f,
                              max_depth = 3,
                              impurity = True,
                              feature_names = list(X_train),
                              class_names = ['Died', 'Survived'],
                              rounded = True,
                              filled= True )

check_call(['dot','-Tpng','tree1.dot','-o','tree1.png'])

In [21]:
img = Image.open("tree1.png")
draw = ImageDraw.Draw(img)
img.save('sample-out.png')
PImage("sample-out.png")